In [30]:
# Fix numpy compatibility issue with gensim
import warnings
warnings.filterwarnings('ignore', message='.*numpy.dtype size changed.*')
warnings.filterwarnings('ignore', message='.*binary incompatibility.*')
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.decomposition import PCA
import os
import csv
import re

# Try to import with error handling
try:
    from utils_5 import _read_dictionary
    print("✅ Utils import successful!")
except ImportError as e:
    print(f"❌ Utils import error: {e}")

# Handle gensim import separately to avoid numpy compatibility issues
try:
    # Try to suppress the specific numpy compatibility warning
    import os
    os.environ['PYTHONWARNINGS'] = 'ignore::UserWarning'
    
    from gensim.models import KeyedVectors
    GENSIM_AVAILABLE = True
    print("✅ Gensim import successful!")
except Exception as e:
    print(f"❌ Gensim import error: {e}")
    print("Word embeddings will use fallback mode (zero embeddings).")
    GENSIM_AVAILABLE = False
    
    # Create a mock KeyedVectors class for fallback
    class MockKeyedVectors:
        def load_word2vec_format(self, *args, **kwargs):
            raise NotImplementedError("Gensim not available - use fallback embeddings")
    
    KeyedVectors = MockKeyedVectors

# Define word embedding function locally to avoid import issues
def sep_by_uppercase(str):
    pattern = "[A-Z]"
    new_str = re.sub(pattern, lambda x: " " + x.group(0), str)
    if str[0].islower():  # first character is lowercase
        return new_str
    else:
        return new_str[1:]

def word_embedding(embeding_file, nodes_dict):
    if not GENSIM_AVAILABLE:
        print("Warning: Gensim not available, returning zero embeddings")
        dim = 300
        return np.zeros((len(nodes_dict), dim))
    
    dim = 300
    embeddings = KeyedVectors.load_word2vec_format(embeding_file, binary=True)
    node_embeding = np.zeros((len(nodes_dict),dim))  # default value is 0
    for nod in nodes_dict:
        words = nod.strip().split('#') # [1][:-1]
        if len(words) == 1:
            words = words[0].split('/')[-1][:-1]
        else:
            words = words[1][:-1]
        words = sep_by_uppercase(words)
        if words in embeddings:
            node_embeding[nodes_dict[nod]] = embeddings[words]
        else:
            if len(words) > 1:  # multi-words
                words = words.split(' ')
                vectors = np.zeros((len(words),dim))
                for w in words:
                    if w in embeddings:
                        vectors[words.index(w)] = embeddings[w]
                node_embeding[nodes_dict[nod]] = np.mean(np.array(vectors), axis=0)
    return node_embeding

print("All functions defined successfully!")


✅ Utils import successful!
✅ Gensim import successful!
All functions defined successfully!


In [31]:

#train_test_idx starts here

In [32]:
#Parameters Initialization
fold = 1
ftype = 'train'
dim = 300
path = '/Users/senurafernando/Documents/GitHub/GCN-based-Ontology-Completion-Fork/dataset/transport/10_fold/'


In [33]:
train_path = path + 'train/s' + str(fold + 1)
train_nodes_label = pd.read_csv(train_path + '/predicate_label.tsv', sep='\t', encoding='utf-8')
node_dict_file = train_path + '/nodes.dict'



print("Train path: ", train_path ) 
print("Train nodes label: ", train_nodes_label)
print("Node dict file: ", node_dict_file)






Train path:  /Users/senurafernando/Documents/GitHub/GCN-based-Ontology-Completion-Fork/dataset/transport/10_fold/train/s2
Train nodes label:        id                                              nodes  \
0      1  <http://reliant.teknowledge.com/DAML/Transport...   
1      2  <http://reliant.teknowledge.com/DAML/Transport...   
2      3  <http://reliant.teknowledge.com/DAML/SUMO.owl#...   
3      4  <http://reliant.teknowledge.com/DAML/Mid-level...   
4      5  <http://reliant.teknowledge.com/DAML/Transport...   
..   ...                                                ...   
778  779  <http://reliant.teknowledge.com/DAML/Transport...   
779  780  <http://reliant.teknowledge.com/DAML/SUMO.owl#...   
780  781  <http://reliant.teknowledge.com/DAML/Transport...   
781  782  <http://reliant.teknowledge.com/DAML/Transport...   
782  783  <http://reliant.teknowledge.com/DAML/Transport...   

                                                 label  
0    TempateExpression{body=[?], head=[<http

In [34]:
train_nodes_label.head(50)

,id,nodes,label
0,1,<http://reliant.teknowledge.com/DAML/Transport...,"TempateExpression{body=[?], head=[<http://reli..."
1,2,<http://reliant.teknowledge.com/DAML/Transport...,"TempateExpression{body=[?], head=[<http://reli..."
2,3,<http://reliant.teknowledge.com/DAML/SUMO.owl#...,TempateExpression{body=[<http://reliant.teknow...
3,4,<http://reliant.teknowledge.com/DAML/Mid-level...,"TempateExpression{body=[?], head=[<http://reli..."
4,5,<http://reliant.teknowledge.com/DAML/Transport...,"TempateExpression{body=[?], head=[<http://reli..."
5,6,<http://reliant.teknowledge.com/DAML/Transport...,TempateExpression{body=[<http://reliant.teknow...
6,7,<http://reliant.teknowledge.com/DAML/Mid-level...,TempateExpression{body=[<http://reliant.teknow...
7,8,<http://reliant.teknowledge.com/DAML/Transport...,TempateExpression{body=[<http://reliant.teknow...
8,9,<http://reliant.teknowledge.com/DAML/SUMO.owl#...,TempateExpression{body=[<http://reliant.teknow...
9,10,<http://reliant.teknowledge.com/DAML/Transport...,"TempateExpression{body=[?], head=[<http://reli..."


In [14]:
if os.path.exists(node_dict_file):
    nodes_dict = _read_dictionary(node_dict_file)
else:
    # node dict
    nodes_dict = dict()
    nodes = train_nodes_label['nodes'].values.tolist() # Loading Nodes from Predicate_Label file
    nod_id = 0
    node_id = ''
    for n in nodes:
        if n not in nodes_dict:
            nodes_dict[n] = nod_id
            node_id += str(nod_id) + '\t' + n + '\n'
            nod_id += 1
    f = open(node_dict_file, 'w', encoding='utf-8')
    f.write(node_id)
    f.close()

print('Number of training nodes: ', len(nodes_dict))

Number of training nodes:  392


'/binary_template_triples_uni.txt' & '/binary_template_triples_bi.txt' datasets have unary and binary templates. Basically nodes and edges are extracted using both templates files and represent as knowledge graph

In [15]:
node_id_con = set()
relation_dict = dict()
rid = 1  # 0 for self-relation, last id + 1 for relation to node "top"
edge_list = []

In [35]:
# uni-edges
f = open(train_path + '/binary_template_triples_uni.txt', 'r', encoding='utf-8')
uni_lines = f.readlines()
f.close()

In [36]:
for line in uni_lines:
  
  triple = line.strip().split('\t')

  print(triple[1])

TempateExpression{body=[?], head=[?, <http://reliant.teknowledge.com/DAML/Transportation.owl#NavigationLight>, <http://reliant.teknowledge.com/DAML/Transportation.owl#TrafficLight>, <http://reliant.teknowledge.com/DAML/Transportation.owl#ElectrifiedRailwayCar>, <http://reliant.teknowledge.com/DAML/Transportation.owl#RadioNavigationBeacon>, <http://reliant.teknowledge.com/DAML/Mid-level-ontology.owl#ElectricMotor>]}
TempateExpression{body=[?], head=[?, <http://reliant.teknowledge.com/DAML/Transportation.owl#NavigationLight>, <http://reliant.teknowledge.com/DAML/Transportation.owl#TrafficLight>, <http://reliant.teknowledge.com/DAML/Transportation.owl#ElectrifiedRailwayCar>, <http://reliant.teknowledge.com/DAML/Transportation.owl#RadioNavigationBeacon>, <http://reliant.teknowledge.com/DAML/Mid-level-ontology.owl#ElectricMotor>]}
TempateExpression{body=[?], head=[?, <http://reliant.teknowledge.com/DAML/Transportation.owl#NavigationLight>, <http://reliant.teknowledge.com/DAML/Transportation

In [38]:
relation_dict 

{'TempateExpression{body=[?], head=[?, <http://reliant.teknowledge.com/DAML/Transportation.owl#NavigationLight>, <http://reliant.teknowledge.com/DAML/Transportation.owl#TrafficLight>, <http://reliant.teknowledge.com/DAML/Transportation.owl#ElectrifiedRailwayCar>, <http://reliant.teknowledge.com/DAML/Transportation.owl#RadioNavigationBeacon>, <http://reliant.teknowledge.com/DAML/Mid-level-ontology.owl#ElectricMotor>]}': 1,
 'TempateExpression{body=[?, <http://reliant.teknowledge.com/DAML/Transportation.owl#Kayak>, <http://reliant.teknowledge.com/DAML/Transportation.owl#Canoe>], head=[?, <http://reliant.teknowledge.com/DAML/Transportation.owl#Axle>, <http://reliant.teknowledge.com/DAML/Mid-level-ontology.owl#HoistingDevice>, <http://reliant.teknowledge.com/DAML/Transportation.owl#TransportationControlDevice>, <http://reliant.teknowledge.com/DAML/Mid-level-ontology.owl#UserPoweredDevice>, <http://reliant.teknowledge.com/DAML/Transportation.owl#CanalLockGate>, <http://reliant.teknowledge.c

In [ ]:
for line in uni_lines:
    triple = line.strip().split('\t')
    if triple[0] in nodes_dict and triple[2] in nodes_dict:
        if triple[1] not in relation_dict:
            relation_dict[triple[1]] = rid
            rid = rid + 1
        src = nodes_dict[triple[0]]
        dst = nodes_dict[triple[2]]
        node_id_con.add(src)
        node_id_con.add(dst)
        edge_list.append((src, dst, relation_dict[triple[1]]))


In [19]:
# bi-edges
f = open(train_path + '/binary_template_triples_bi.txt', 'r', encoding='utf-8')
lines = f.readlines()
f.close()

In [20]:
for line in lines:
    triple = line.strip().split('\t')
    if triple[0] in node_id_con and triple[2] in node_id_con:
        if triple[1] not in relation_dict:
            relation_dict[triple[1]] = rid
            rid = rid + 1
        src = nodes_dict[triple[0]]
        dst = nodes_dict[triple[2]]
        node_id_con.add(src)
        node_id_con.add(dst)
        edge_list.append((src, dst, relation_dict[triple[1]]))
        edge_list.append((dst, src, relation_dict[triple[1]]))

In [21]:
# self-connection edge
for n in node_id_con:  # the graph only includes nodes with edges
    edge_list.append((n, n, 0))

# sort indices by destination
edge_list = sorted(edge_list, key=lambda x: (x[1], x[0], x[2]))
edge_list = np.array(edge_list, dtype=np.int32)

In [22]:
# unary templates (node labels)
label_dict_file = train_path + '/unary_templates.dict'
if os.path.exists(label_dict_file):
    label_dict = _read_dictionary(label_dict_file)
else:
    label_dict = dict()
    l_id = 0
    label_id = ''
    for nod, lab in zip(train_nodes_label['nodes'].values, train_nodes_label['label'].values):
        if nod not in nodes_dict:
            continue
        lab = lab.replace(',TempateExpression', '\tTempateExpression').split('\t')
        if len(lab) > 1:
            for l in lab:
                if l not in label_dict:
                    label_dict[l] = l_id
                    label_id += str(l_id) + '\t' + l + '\n'
                    l_id = l_id + 1
        else:
            if lab[0] not in label_dict:
                label_dict[lab[0]] = l_id
                label_id += str(l_id) + '\t' + lab[0] + '\n'
                l_id = l_id + 1
    f = open(label_dict_file, 'w', encoding='utf-8')
    f.write(label_id)
    f.close()

In [23]:
num_node = len(nodes_dict)
num_rel = len(relation_dict) + 1
train_labels = sp.lil_matrix((num_node, len(label_dict)))
for nod, lab in zip(train_nodes_label['nodes'].values, train_nodes_label['label'].values):
    if nod not in nodes_dict:
        continue
    nod_id = nodes_dict[nod]
    lab = lab.replace(',TempateExpression', '\tTempateExpression').split('\t')
    if len(lab) > 1:
        for l in lab:
            lab_id = label_dict[l]
            train_labels[nod_id, lab_id] = 1
    else:
        lab_id = label_dict[lab[0]]
        train_labels[nod_id, lab_id] = 1

In [24]:
train_labels = train_labels.tocsr().todense()

In [25]:
train_labels

matrix([[1., 1., 0., ..., 0., 0., 0.],
        [0., 0., 1., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 1., 1., 1.]])

In [26]:
test_path = path + 'test/s' + str(fold + 1)
test_nodes_label = pd.read_csv(test_path + '/predicate_label.tsv', sep='\t', encoding='utf-8')
test_nodes = test_nodes_label['nodes']
test_idx = []
for n in test_nodes:
    if n in nodes_dict and nodes_dict[n] in node_id_con:
        test_idx.append(nodes_dict[n])
test_idx = sorted(list(set(test_idx)))

test_labels = sp.lil_matrix((len(test_idx), len(label_dict)))
i = 0
for nod, lab in zip(test_nodes_label['nodes'].values, test_nodes_label['label'].values):
    if nod not in nodes_dict:
        i += 1
        continue
    n_id = nodes_dict[nod]
    if n_id not in test_idx:
        i += 1
        continue
    j = test_idx.index(n_id)
    lab = lab.replace(',TempateExpression', '\tTempateExpression').split('\t')
    if len(lab) > 1:
        for l in lab:
            if l in label_dict:
                lab_id = label_dict[l]
                test_labels[j, lab_id] = 1
                # print(test_idx, nodes_dict[nod], j, test_labels[j])
    else:
        if lab[0] in label_dict:
            lab_id = label_dict[lab[0]]
            test_labels[j, lab_id] = 1
print(i, 'test nodes not in training set')

test_labels = test_labels.tocsr()

82 test nodes not in training set


In [27]:
# remove nodes that all the labels are 0
idx = np.where(test_labels.toarray().sum(axis=1) > 0)[0]
test_labels = test_labels[idx]
test_idx = np.array(test_idx)[idx]
test_labels = test_labels.tocsr()

In [29]:
ftype = 'embedding'
if ftype == 'embedding':
    feature_file = train_path + '/em_features.csv'
elif ftype == 'analogy':
    feature_file = train_path + '/an_features_' + str(dim) + '.csv'

if not os.path.exists(feature_file):
    if ftype == 'embedding':
        embedding_file = 'dataset/GoogleNews-vectors-negative300.bin.gz'
        node_features = word_embedding(embedding_file, nodes_dict)
    elif ftype == 'analogy':
        pca = PCA(n_components=dim)
        node_features = pca.fit_transform(train_labels)
    f = open(feature_file, 'w', encoding='utf-8', newline='')
    writer = csv.writer(f)
    writer.writerow(node_features[0])
    writer.writerows(node_features)
    f.close()
else:
    node_features = pd.read_csv(feature_file, sep=',', encoding='utf-8')

edge_src, edge_dst, edge_type = edge_list.transpose()
_, inverse_index, count = np.unique((edge_dst, edge_type), axis=1, return_inverse=True,
                                    return_counts=True)
degrees = count[inverse_index]  # c_{i,r} for each relation type
edge_norm = np.ones(len(edge_dst), dtype=np.float32) / degrees.astype(np.float32)

FileNotFoundError: [Errno 2] No such file or directory: 'dataset/GoogleNews-vectors-negative300.bin.gz'